<a href="https://colab.research.google.com/github/paolobalasso/DataScienceCourse/blob/main/notebooks/05_conjoint_demand_revenue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 05 — Conjoint Analysis → Potential Demand → Revenue

**End-to-end pipeline** in one notebook:
1. **Design** a choice-based conjoint study (attributes, levels, choice sets).
2. **Simulate respondents** with heterogeneous part-worths (so the data behaves like a real CBC survey).
3. **Estimate** an aggregate **Multinomial Logit (MNL)** to recover part-worths.
4. Derive **attribute importance** and **willingness-to-pay (WTP)**.
5. Build a **market simulator** — share-of-preference for any product configuration.
6. Translate share → **potential demand** (units) given a market size.
7. Compute **revenue** and find the **revenue-maximising price**.

*Didactic notebook — synthetic data, ~30 s on a Colab CPU. The math is real, the numbers are made up.*


## 0. Setup

In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn statsmodels
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from itertools import product
import statsmodels.api as sm
rng = np.random.default_rng(7)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True


## 1. Conjoint design

We are pricing a **streaming subscription**. Four attributes:

| Attribute | Levels |
|---|---|
| `Price` (EUR/month) | 7.99, 11.99, 15.99 |
| `Catalog` | Basic, Standard, Premium |
| `Ads` | With ads, No ads |
| `Devices` | 1 device, 2 devices, 4 devices |

Full factorial = 3·3·2·3 = **54 profiles**. Each respondent sees **8 choice tasks**, each task offering **3 alternatives + a 'None'** option.


In [ ]:
PRICE   = [7.99, 11.99, 15.99]
CATALOG = ['Basic', 'Standard', 'Premium']
ADS     = ['With ads', 'No ads']
DEVICES = [1, 2, 4]

profiles = pd.DataFrame(list(product(PRICE, CATALOG, ADS, DEVICES)),
                        columns=['price','catalog','ads','devices'])
print(f'Full factorial: {len(profiles)} profiles')
profiles.head()


## 2. Simulate 400 respondents with heterogeneous part-worths

True (population-mean) utilities — we will try to recover these in step 3:

- price: **-0.25 per EUR** (linear)
- catalog: Basic=0 (base), Standard=+0.6, Premium=+1.1
- ads: With ads=0 (base), No ads=+0.8
- devices: 1=0, 2=+0.4, 4=+0.7
- 'None' option intercept: **-2.0**

Each respondent gets these means + Gaussian noise → realistic individual heterogeneity.


In [ ]:
N_RESP = 400
N_TASKS = 8
N_ALTS = 3   # plus 'None'

# population-level true part-worths (what MNL should recover, up to noise)
TRUE = {'price': -0.25,
        'cat_Standard': 0.6, 'cat_Premium': 1.1,
        'ads_No': 0.8,
        'dev_2': 0.4, 'dev_4': 0.7,
        'none': -2.0}

# individual-level part-worths = TRUE + noise
def draw_individual():
    return {k: v + rng.normal(0, 0.25) for k, v in TRUE.items()}

def utility(profile, betas):
    u = betas['price'] * profile['price']
    if profile['catalog']=='Standard': u += betas['cat_Standard']
    if profile['catalog']=='Premium':  u += betas['cat_Premium']
    if profile['ads']=='No ads':       u += betas['ads_No']
    if profile['devices']==2:          u += betas['dev_2']
    if profile['devices']==4:          u += betas['dev_4']
    return u

rows = []
for r in range(N_RESP):
    betas = draw_individual()
    for t in range(N_TASKS):
        # sample 3 distinct profiles for this task
        task_profiles = profiles.sample(N_ALTS, random_state=int(rng.integers(1e9))).reset_index(drop=True)
        utils = [utility(p, betas) for _, p in task_profiles.iterrows()] + [betas['none']]
        # add Gumbel noise -> multinomial logit
        gumbel = -np.log(-np.log(rng.uniform(size=N_ALTS+1)))
        choice = int(np.argmax(np.array(utils) + gumbel))
        for alt_idx in range(N_ALTS):
            p = task_profiles.iloc[alt_idx]
            rows.append({'resp': r, 'task': t, 'alt': alt_idx,
                         'price': p['price'], 'catalog': p['catalog'],
                         'ads': p['ads'], 'devices': p['devices'],
                         'is_none': 0,
                         'chosen': int(choice == alt_idx)})
        # 'None' alternative
        rows.append({'resp': r, 'task': t, 'alt': N_ALTS,
                     'price': 0.0, 'catalog': 'NONE', 'ads': 'NONE', 'devices': 0,
                     'is_none': 1,
                     'chosen': int(choice == N_ALTS)})

cbc = pd.DataFrame(rows)
print(cbc.shape)
cbc.head(8)


## 3. Estimate aggregate MNL

We fit a logit on the chosen-vs-not-chosen panel. Because conjoint data has one row per alternative and one chosen alternative per task, the standard trick is to estimate via **conditional logit** — equivalent here to a logistic regression of `chosen` on the alternative attributes *with task fixed effects absorbed* (the choice-set normalisation).

For a teaching notebook we use **plain logistic regression on the alternative-level rows** — fine for recovering relative part-worths when alternatives within a task are i.i.d. sampled (which they are, by design).


In [ ]:
# build design matrix
X = pd.DataFrame({
    'price':       cbc['price'],
    'cat_Standard':(cbc['catalog']=='Standard').astype(int),
    'cat_Premium': (cbc['catalog']=='Premium').astype(int),
    'ads_No':      (cbc['ads']=='No ads').astype(int),
    'dev_2':       (cbc['devices']==2).astype(int),
    'dev_4':       (cbc['devices']==4).astype(int),
    'none':        cbc['is_none'],
})
y = cbc['chosen']

X_const = sm.add_constant(X, has_constant='add')
model = sm.Logit(y, X_const).fit(disp=False)
print(model.summary().tables[1])


In [ ]:
# compare estimated vs true
est = model.params.drop('const')
comp = pd.DataFrame({'true': pd.Series(TRUE), 'estimated': est}).round(3)
comp['gap'] = (comp['estimated'] - comp['true']).round(3)
comp


**Read the gap column.** Small gaps (≈ ±0.1) mean the simulator did its job and the MNL recovered the design. If you re-run with `N_RESP=50` you will see the gap blow up — sample size matters.


## 4. Attribute importance & willingness-to-pay

**Importance** = range of part-worths within an attribute, normalised across all attributes (textbook conjoint convention).

**WTP** for moving from level A to level B = (β_B − β_A) / (−β_price). Read it as 'how many extra EUR/month a customer would pay to get level B instead of A'.


In [ ]:
b = est.to_dict()
# build ranges per attribute
ranges = {
    'price':   abs(b['price'] * (max(PRICE) - min(PRICE))),
    'catalog': max(0, b['cat_Standard'], b['cat_Premium']) - min(0, b['cat_Standard'], b['cat_Premium']),
    'ads':     abs(b['ads_No']),
    'devices': max(0, b['dev_2'], b['dev_4']) - min(0, b['dev_2'], b['dev_4']),
}
total = sum(ranges.values())
importance = {k: round(100*v/total,1) for k, v in ranges.items()}
print('Attribute importance (%):', importance)

fig, ax = plt.subplots(figsize=(6,3.2))
ax.barh(list(importance.keys()), list(importance.values()), color='#1B263B')
ax.set_xlabel('Importance (%)'); ax.set_title('Attribute importance')
plt.tight_layout(); plt.show()


In [ ]:
# Willingness to pay
wtp = {
    'Standard vs Basic':  -b['cat_Standard'] / b['price'],
    'Premium vs Basic':   -b['cat_Premium']  / b['price'],
    'No ads vs With ads': -b['ads_No']       / b['price'],
    '2 devices vs 1':     -b['dev_2']        / b['price'],
    '4 devices vs 1':     -b['dev_4']        / b['price'],
}
pd.Series(wtp).round(2).to_frame('WTP (EUR/month)')


**How to read:** if 'No ads vs With ads' = 3.20 EUR, an average customer would pay €3.20 more per month for an ad-free tier. Compare this to the price gap your CFO wants to charge — that is the negotiation.


## 5. Market simulator — share of preference

Given a set of competing products (incl. 'None'), MNL gives the **share of preference**:
$$\\text{share}_i = \\frac{e^{U_i}}{\\sum_j e^{U_j}}$$

Define a competitive scenario:
- **Our product (variants)**: Premium / No ads / 4 devices, price varied 7.99 → 19.99
- **Competitor A**: Standard / With ads / 2 devices @ 9.99
- **Competitor B**: Premium / With ads / 4 devices @ 13.99
- **None** (status quo)


In [ ]:
def util_row(price, catalog, ads, devices):
    u = b['price'] * price
    if catalog=='Standard': u += b['cat_Standard']
    if catalog=='Premium':  u += b['cat_Premium']
    if ads=='No ads':       u += b['ads_No']
    if devices==2:          u += b['dev_2']
    if devices==4:          u += b['dev_4']
    return u

comp_A = util_row(9.99,  'Standard', 'With ads', 2)
comp_B = util_row(13.99, 'Premium',  'With ads', 4)
u_none = b['none']

prices = np.arange(7.99, 20.0, 0.5)
shares_us, shares_A, shares_B, shares_none = [], [], [], []
for p_us in prices:
    u_us = util_row(p_us, 'Premium', 'No ads', 4)
    utils = np.array([u_us, comp_A, comp_B, u_none])
    exp_u = np.exp(utils - utils.max())
    s = exp_u / exp_u.sum()
    shares_us.append(s[0]); shares_A.append(s[1])
    shares_B.append(s[2]); shares_none.append(s[3])

fig, ax = plt.subplots(figsize=(7,4))
ax.stackplot(prices, shares_us, shares_A, shares_B, shares_none,
             labels=['Us (Premium/No ads/4)', 'Competitor A', 'Competitor B', 'None'],
             colors=['#D4A017','#5DADE2','#58D68D','#BDC3C7'], alpha=0.85)
ax.set_xlabel('Our price (EUR/month)'); ax.set_ylabel('Share of preference')
ax.set_title('Market share vs our price')
ax.legend(loc='lower left', fontsize=8); plt.tight_layout(); plt.show()


## 6. Potential demand & revenue

Assume the addressable market is **M = 2,000,000 households** in our country. Potential demand at each price = share × M. Revenue = demand × price.


In [ ]:
M = 2_000_000
demand = np.array(shares_us) * M
revenue = demand * prices

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11,3.8))
ax1.plot(prices, demand/1e3, color='#1B263B', lw=2)
ax1.set_xlabel('Price (EUR/month)'); ax1.set_ylabel('Potential demand (k households)')
ax1.set_title('Demand curve')

ax2.plot(prices, revenue/1e6, color='#D4A017', lw=2)
opt_idx = int(np.argmax(revenue))
ax2.axvline(prices[opt_idx], color='red', ls='--', alpha=0.6,
            label=f'Optimal: {prices[opt_idx]:.2f} EUR')
ax2.set_xlabel('Price (EUR/month)'); ax2.set_ylabel('Monthly revenue (EUR, M)')
ax2.set_title('Revenue curve'); ax2.legend()
plt.tight_layout(); plt.show()

print(f'Revenue-maximising price:  {prices[opt_idx]:.2f} EUR/month')
print(f'Potential demand at opt:   {demand[opt_idx]:,.0f} households')
print(f'Monthly revenue at opt:    {revenue[opt_idx]/1e6:,.2f}M EUR')
print(f'Annualised revenue at opt: {revenue[opt_idx]*12/1e6:,.2f}M EUR')


## 7. Sensitivity — what if we got the part-worths wrong?

Bootstrap the choice data, refit MNL, re-compute optimal price. The spread is the **decision-relevant uncertainty**.


In [ ]:
def fit_and_optimise(cbc_sample):
    Xs = pd.DataFrame({
        'price':       cbc_sample['price'],
        'cat_Standard':(cbc_sample['catalog']=='Standard').astype(int),
        'cat_Premium': (cbc_sample['catalog']=='Premium').astype(int),
        'ads_No':      (cbc_sample['ads']=='No ads').astype(int),
        'dev_2':       (cbc_sample['devices']==2).astype(int),
        'dev_4':       (cbc_sample['devices']==4).astype(int),
        'none':        cbc_sample['is_none'],
    })
    ys = cbc_sample['chosen']
    m = sm.Logit(ys, sm.add_constant(Xs, has_constant='add')).fit(disp=False)
    bb = m.params.drop('const').to_dict()
    def U(price, catalog, ads, devices):
        u = bb['price']*price
        if catalog=='Standard': u += bb['cat_Standard']
        if catalog=='Premium':  u += bb['cat_Premium']
        if ads=='No ads':       u += bb['ads_No']
        if devices==2:          u += bb['dev_2']
        if devices==4:          u += bb['dev_4']
        return u
    cA = U(9.99,'Standard','With ads',2); cB = U(13.99,'Premium','With ads',4); cN = bb['none']
    best_p, best_rev = None, -1
    for p in prices:
        u_us = U(p,'Premium','No ads',4)
        utils = np.array([u_us, cA, cB, cN])
        exp_u = np.exp(utils - utils.max()); s = exp_u/exp_u.sum()
        rev = s[0]*M*p
        if rev > best_rev: best_rev, best_p = rev, p
    return best_p, best_rev

resp_ids = cbc['resp'].unique()
boot_prices, boot_revs = [], []
for _ in range(40):
    sample_resp = rng.choice(resp_ids, size=len(resp_ids), replace=True)
    sample = pd.concat([cbc[cbc['resp']==r] for r in sample_resp], ignore_index=True)
    bp, br = fit_and_optimise(sample)
    boot_prices.append(bp); boot_revs.append(br)

print(f'Optimal price 90% CI: [{np.percentile(boot_prices,5):.2f}, {np.percentile(boot_prices,95):.2f}] EUR')
print(f'Revenue       90% CI: [{np.percentile(boot_revs,5)/1e6:.2f}, {np.percentile(boot_revs,95)/1e6:.2f}] M EUR')

fig, ax = plt.subplots(figsize=(6,3))
ax.hist(boot_prices, bins=12, color='#1B263B', alpha=0.85)
ax.set_xlabel('Bootstrapped optimal price'); ax.set_title('Decision uncertainty')
plt.tight_layout(); plt.show()


## 8. Mini-exercises

1. **Add a 4th level** to `Catalog` (e.g. `Premium+Sports`). Re-simulate with a true part-worth of +1.5. Does MNL recover it?
2. **Change the competitor mix**: drop Competitor B, add a free ad-supported tier (price=0, with ads). What happens to the optimal price?
3. **Heterogeneous segments**: split respondents into 'students' (price coefficient −0.4) and 'families' (price −0.15). Re-fit MNL globally vs per-segment. Where does the aggregate model lie? (Hint: it is *not* the average.)
4. **Profit instead of revenue**: assume marginal cost = €3/month. Optimise `share × M × (price − 3)`. How far does the optimal price shift?

## 9. Caveats — what this notebook does NOT do

- **No latent-class / mixed logit.** Real CBC analysis recovers heterogeneity with HB (Hierarchical Bayes) or LC-MNL. Aggregate MNL biases the price coefficient towards zero (IIA + heterogeneity).
- **No purchase-frequency model.** Share of preference ≠ probability of purchase in a given month. A real revenue model needs choice × frequency × retention.
- **Static competitors.** Reality: competitors react. Equilibrium pricing is a game, not an optimisation.
- **No price endogeneity.** Stated-preference vs revealed-preference gap is real; in deployment you would calibrate WTP against an A/B price test.
- **No discrete-choice efficiency design.** We use random profiles per task; a real CBC uses balanced/orthogonal or D-efficient designs.

## Cross-references

- **Slides**: Main FIX8 s67-70 (price elasticity, log-log).
- **Book chapter**: Ch 12 — Pricing.
- **Related notebook**: `03_demand_eda_multi_product.ipynb` — same underlying logic when SKUs cannibalise.
